# R1 — the naive numeric rung, in hand-rolled NumPy

The ladder's first rung is deliberately the dumbest thing that can learn:
two cards become numbers, one small feed-forward stack is written by hand in NumPy, and the
agent plays heads-up fixed-limit Hold'em against a frozen league of its own past versions
plus three fixed baselines.

Everything below is a call into the repository: `training` for the rung, the run and the
checkpoints, `evaluation` for the league table, the rating and the report. There is no loop,
no model and no metric in this notebook — if a cell ever needs one, the module owes it an
entry point (D-5).

The rung's one axis is the *learning rule's* representation: R1 is the naive numeric encoding,
hand-rolled in NumPy, so the arithmetic is visible (D-7). Later rungs re-implement this same
model in PyTorch (R2), tokenise the cards into a learned embedding (R3), then vary width against
depth (R4), connectivity (R5) and a residual path (R6) — one axis at a time, on one game (D-3).

**Where it runs from.** The notebook finds the repository root itself — walking up from the
kernel's working directory to the directory that carries `pyproject.toml` and `src/` — so it
runs whether the kernel starts in `notebooks/` (which is what `jupyter lab
notebooks/r1-naive-numpy.ipynb` does) or at the repository root. Every path a record spells
repository-relative is resolved against that root before it is handed back to `evaluation`,
because `evaluation.policy_from` and `evaluation.build_pool` test a path against the process's
working directory.

**Where it writes.** The run below writes ten snapshots and a final model per seed — 33 files
for R1's three seeds — and at the declared seeds and budget those are exactly the paths the
repository's shipped set occupies. Executed as `README.md` §Quickstart shows, this notebook
therefore rewrote all 33 shipped checkpoint files in place during walkthrough run 3: identical
payloads on that build, but on any other build the shipped models would be silently replaced.
So the first code cell sets its own write root before it trains — `/tmp/pypoker-notebook-r1`,
unless `PYPOKER_CHECKPOINT_ROOT` is already set — and the shipped set stays where it is. Set
`PYPOKER_CHECKPOINT_ROOT` yourself to write anywhere else, or to an absolute path inside
`checkpoints/` if you do mean to replace the shipped models (`docs/L1/ladder.md` §7,
`docs/L1/getting-started.md` §4).

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt


def _repository_root(start):
    """The tree this notebook ships in: the nearest ancestor carrying pyproject.toml and src/."""
    here = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(here, "pyproject.toml")) and os.path.isdir(
            os.path.join(here, "src")
        ):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            raise RuntimeError(
                "no repository root above %r: this notebook reads the tree it ships in"
                % (start,)
            )
        here = parent


ROOT = _repository_root(os.getcwd())
if os.path.join(ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, "src"))


def resolve(path):
    """A path a record spells repository-relative, read from the repository root."""
    text = os.fspath(path) if not isinstance(path, str) else str(path)
    return text if os.path.isabs(text) else os.path.join(ROOT, text)


# The run below writes 33 files. Point that write root away from the shipped set unless the
# caller already named one: PYPOKER_CHECKPOINT_ROOT is what training.layout reads, on every call.
SHIPPED_ROOT = os.path.join(ROOT, "checkpoints")
WRITE_ROOT = os.environ.get("PYPOKER_CHECKPOINT_ROOT") or "/tmp/pypoker-notebook-r1"
os.environ["PYPOKER_CHECKPOINT_ROOT"] = WRITE_ROOT

import training
import evaluation

RUNG = "R1"
print("repository root:", ROOT)
print("kernel working directory:", os.getcwd())
print("the ladder:", list(training.rungs()))
print("the device:", training.devices()["default"], "|", training.devices()["gpu"]["reason"])
print(
    "run writes under:", WRITE_ROOT,
    "| the shipped set is replaced by this run"
    if os.path.abspath(WRITE_ROOT) == os.path.abspath(SHIPPED_ROOT)
    else "| the shipped set at %s is untouched by this run" % SHIPPED_ROOT,
)

## What the rung declares

A rung is a row of data, not a script: the variant's backend and encoding, the architecture's
settings, the sample and parameter budgets, the seed set, the table budget and the frozen pool
it inherits. R1 inherits nothing, so its pool is the three pinned baselines alone.

In [ ]:
config = training.rung(RUNG)
print("rung:", config["name"], "| axis:", config["axis"])
print("primary:", config["primary"], "| carried from:", config["carried_from"])
for variant in config["variants"]:
    print(
        "  variant", variant["name"],
        "| backend", variant["backend"],
        "| encoding", variant["encoding"],
        "| architecture", variant["architecture"],
    )
print("pool:", config["pool"]["id"], "| bound M:", config["pool"]["m"],
      "| versions in the window:", len(config["pool"]["members"]))
print("seeds:", config["seeds"], "| hands per variant per seed:", config["samples"],
      "| table budget:", config["table"])

## The live run

One call. The loop freezes the pool before the first hand, plays each hand through the engine
against an opponent drawn from that pool, learns once per hand on the learner's own decisions,
writes ten equally spaced snapshots, and returns the record of what happened: curves, pool,
checkpoints, seeds, budget, device, declared parameter budget, table budget and the measured
cost per hand. Nothing in it claims a result.

The live budget is 1,000 hands per variant per seed at seeds 0, 1 and 2 — a few minutes on CPU.

It writes under the root the cell above named — `/tmp/pypoker-notebook-r1` by default — so the
repository's shipped `checkpoints/` set is not the run's target.

In [ ]:
record = training.run(RUNG)
print("rung:", record["rung"], "| seeds:", record["seeds"],
      "| hands per variant per seed:", record["budget"], "| device:", record["device"])
for variant, block in record["timings"].items():
    for seed, facts in block.items():
        print(
            "%s seed %s: %d hands in %.1f s — %.1f ms/hand, %.2f hands/s"
            % (variant, seed, facts["hands"], facts["wall_clock_seconds"],
               1000.0 * facts["seconds_per_hand"], facts["hands_per_second"])
        )
print("pool trained against:", record["pool"]["id"], "| frozen:", record["pool"]["frozen"])
print("pool members:", [member["id"] for member in record["pool"]["members"]])

## The training curves

The learning signal is the update's own mean absolute temporal-difference error, averaged over
the hands inside each snapshot interval. Three seeds, so the record also carries the spread
across seeds at every point: a curve on its own is not progress, and a curve with no seed
spread is not evidence (D-6).

In [ ]:
figure, axes = plt.subplots(figsize=(9, 4))
for variant, block in record["curves"].items():
    for seed in sorted(block):
        points = block[seed]
        axes.plot(points["samples"], points["td_error"], marker="o",
                  label="%s, seed %s" % (variant, seed))
    mean = [sum(block[seed]["td_error"][i] for seed in block) / len(block)
            for i in range(len(next(iter(block.values()))["td_error"]))]
    spread = next(iter(block.values()))["spread"]
    samples = next(iter(block.values()))["samples"]
    axes.fill_between(samples, [m - s for m, s in zip(mean, spread)],
                      [m + s for m, s in zip(mean, spread)], alpha=0.15,
                      label="spread across seeds")
axes.set_xlabel("hands trained")
axes.set_ylabel("mean |TD error| per snapshot interval")
axes.set_title("R1 — the naive NumPy rung: one curve per seed, the band is the spread")
axes.legend(fontsize=8)
plt.tight_layout()
plt.show()

for variant, block in record["curves"].items():
    for seed in sorted(block):
        print(variant, "seed", seed, "td_error:",
              [round(value, 3) for value in block[seed]["td_error"]])
    print(variant, "spread:", [round(value, 3) for value in
                                next(iter(block.values()))["spread"]])

## The league table over the pool it trained against

The same frozen pool is the evaluation opposition (D-16). The rung's own final model of seed 0
enters the table as a member beside the three pinned baselines, every cell is 120 hands with the
seats alternating, and the table is reported in both directions — wins, ties, losses, mean chip
delta per hand and each member's measured parameter count.

The rating is Bradley–Terry, fitted over the cells' hand outcomes with ties counted half to each
side, a 95% interval from a seeded hand-level bootstrap, and anchored so the three baselines'
mean is zero: a value reads as a distance from the only fixed standard in the table.

In [ ]:
versions = [
    dict(member, path=resolve(member["path"]))
    for member in record["pool"]["members"]
    if member.get("path")
]
pool = evaluation.build_pool(
    {"members": versions, "m": record["pool"]["m"], "frozen": True}
)
variant_name = config["variants"][0]["name"]
final_model = resolve(
    record["checkpoints"][variant_name][str(record["seeds"][0])]["final"]
)
variant_policy = evaluation.policy_from(final_model)

matrix = evaluation.matrix(pool, [variant_policy], "live")
rating = evaluation.rating(matrix)

print("table:", matrix["table"], "|", len(matrix["members"]), "members |",
      len(matrix["cells"]), "cells at", matrix["hands_per_cell"], "hands per cell |",
      "table seed", matrix["table_seed"])
print()
for cell in matrix["cells"]:
    left, right = cell["members"]
    row = cell["directions"][0]
    print("%-12s vs %-12s %4d W %3d T %4d L   mean chip delta %+.3f   params %s"
          % (left, right, row["wins"], row["ties"], row["losses"],
             row["mean_chip_delta"], cell["params"]))
print()
print("rating:", rating["method"], "| anchor:", rating["anchor"],
      "| converged:", rating["converged"], "| bound applied:", rating["bound_applied"],
      "| bootstrap seed:", rating["bootstrap_seed"])
for member, entry in rating["values"].items():
    low, high = entry["interval"]
    print("%-12s %+7.3f   95%% [%+7.3f, %+7.3f]   over %d hands"
          % (member, entry["value"], low, high, entry["hands"]))

## The report

`evaluation.report` refuses to assemble a per-rung report unless all seven evidence families are
present and consistent: the curves, the frozen pool, the matrix, the rating, the baseline rows,
the axis row — `n/a, single variant` for a one-variant rung, which is a value and not a blank —
and the header. The run record carries five of the seven; the matrix and the rating are the
table's, so the notebook hands them back to it.

In [ ]:
record["matrix"] = matrix
record["rating"] = rating
report = evaluation.report(RUNG, record, matrix, rating)
print("report families:", sorted(str(key) for key in report))
print("header:", sorted(record["header"]))
print("axis row:", report.get("axis row", record.get("axis row")))

## What ships with the repository

A run writes ten snapshots per variant per seed and names the last five plus that pair's final
model; `training.checkpoints()` is the index over exactly the set the repository ships, declared
as data and never scanned from disk. `tables/league-full.json` is the pre-computed full table
(1,500 hands per cell) that `evaluation` produced offline — read here as plain JSON, because no
entry point reads it (D-8).

In [ ]:
index = training.checkpoints()
print("shipped rungs:", index["shipped"], "| not yet shipped:", index["pending"])
print("index entries:", len(index["entries"]))
for entry in index["entries"][:4]:
    print(" ", entry["kind"], entry["id"])
print("  ...")
print("every listed artefact exists:",
      all(os.path.exists(resolve(entry["path"])) for entry in index["entries"]))
print()
with open(os.path.join(ROOT, "tables", "league-full.json")) as handle:
    shipped = json.load(handle)
print("shipped full table:", shipped["table"], "|", shipped["rating"]["method"],
      "| hands played:", shipped["cost"]["hands"],
      "| measured cost: %.3f s/hand (%.1f hands/s)"
      % (shipped["cost"]["seconds_per_hand"], shipped["cost"]["hands_per_second"]))
for member, entry in shipped["rating"]["values"].items():
    low, high = entry["interval"]
    print("%-12s %+7.3f   95%% [%+7.3f, %+7.3f]" % (member, entry["value"], low, high))

## Reading the numbers honestly

A rung is not progress on a curve alone (D-6), and a league rank is not a claim of poker skill
(D-16). What this notebook shows is what R1 measured on one small game at one cell size: its
learning signal per interval, its pool, and where its final model stands among the three fixed
baselines with a 95% interval around every value. The live table is read coarsely — 120 hands a
cell resolves about ±9 percentage points of win rate — so a small order difference between
members is not resolvable from it; the shipped full table and its intervals are what a claim
rests on.

The next rung is the same model in PyTorch, and its evidence is the *clock*: R1 and R2 declare one
encoding, one architecture, one update rule and one sample order, so they are handed the same
deals and must choose the same actions — and the only thing that may differ is the throughput
(D-7). `training.fairness(training.rung("R1"), training.rung("R2"))` is that comparison, stated
as data rather than as a promise.